# Lab 2: Data Loading, Cleaning, and Exploration

In this lab, we will cover the following topics:
1. Data loading and cleaning
2. Handling missing data
3. Feature engineering examples
4. Visualizations/data exploration

Each section includes basic implementation and questions for further exploration.

## 1. Data Loading and Cleaning

We will start by loading and cleaning a dataset. We will also explore different techniques for data cleaning.

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

def simple_model_safe(input_data, target_column="survived", scale_data=False, print_report=True):
    """
    Logistic Regression model that handles categorical and missing data safely.
    """
    # 1️Drop rows where target is missing
    data = input_data.dropna(subset=[target_column]).copy()
    
    # 2️ Drop columns with too many missing values (optional)
    data = data.drop(columns=["deck"])  # 885 missing, drops risk of errors

    # 3️ Fill remaining missing numeric values with median
    numeric_cols = data.select_dtypes(include="number").columns
    for col in numeric_cols:
        data[col] = data[col].fillna(data[col].median())
    
    # 4️ Split features and target
    X = data.drop(target_column, axis=1)
    y = data[target_column]
    
    # 5️ One-hot encode categorical features
    X = pd.get_dummies(X, drop_first=True)
    
    # 6️ Split train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # 7️ Scale if needed
    if scale_data:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
    
    # 8️ Train logistic regression
    log_reg = LogisticRegression(random_state=42, max_iter=200, solver="liblinear")
    log_reg.fit(X_train, y_train)
    
    # 9️ Evaluate
    y_pred = log_reg.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy:.4f}")
    
    if print_report:
        print("Classification Report:")
        print(classification_report(y_test, y_pred))
    
    return log_reg

In [15]:
import pandas as pd

data = pd.read_csv("messy_data.csv")
model = simple_model_safe(data, target_column="survived", scale_data=True, print_report=True)

Accuracy: 0.8233
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.87      0.86       143
           1       0.79      0.74      0.76        89

    accuracy                           0.82       232
   macro avg       0.82      0.81      0.81       232
weighted avg       0.82      0.82      0.82       232



In [11]:
#import functionals as f
import pandas as pd
import numpy as np

# Load the dataset
path_to_file = 'messy_data.csv'
data = pd.read_csv(path_to_file)

# Display dataset information
data.head()
data.info()
#data.describe()

# Run the simple model
f.simple_model(data)

<class 'pandas.DataFrame'>
RangeIndex: 1158 entries, 0 to 1157
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     1158 non-null   int64  
 1   deck         273 non-null    str    
 2   embarked     1155 non-null   str    
 3   pclass       1051 non-null   float64
 4   embark_town  1155 non-null   str    
 5   sex          1158 non-null   str    
 6   adult_male   1158 non-null   bool   
 7   who          1158 non-null   str    
 8   wspd         929 non-null    float64
 9   lfwa         929 non-null    float64
 10  class        1158 non-null   str    
 11  tprc         1158 non-null   float64
 12  sibsp        1044 non-null   float64
 13  age          829 non-null    float64
 14  alone        1158 non-null   bool   
 15  fare         1036 non-null   float64
 16  parch        1042 non-null   float64
dtypes: bool(2), float64(8), int64(1), str(6)
memory usage: 138.1 KB


c:\Users\navpr\Desktop\Bioinformatics\MAchine Learning\Assignment\BINF-5507_Materilas\.pixi\envs\default\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


ValueError: could not convert string to float: 'C'

## Data Cleaning

We will clean the dataset by removing duplicates, fixing inconsistent entries and removing outliers.

In [ ]:
# Remove duplicates
data_no_duplicates = data.copy().drop_duplicates()

# Fix inconsistent entries
data_format_fixed = data.copy()
data_format_fixed['sex'] = data_format_fixed['sex'].apply(lambda x: 'female' if 'F' in x or 'f' in x else 'male')

# Check for outliers - if they exist, remove them
# <insert code here>


# Dataset with no duplicates, fixed format, missing values and outlier removed (if they exist)
# <insert code here>


### Questions for Exploration

1. How does the following affect model performance?
    * removing duplicates
    * fixing inconsistencies  
2. What other inconsistencies can you find and fix in the dataset?
3. How does the choice of dataset affect the data cleaning process?

## 2. Handling Missing Data

We will handle missing data by using different techniques such as imputation and deletion. We will also explore the impact of these techniques on the dataset.

In [ ]:
# Identify missing values
missing_data = data.isnull().sum()
print(missing_data)

# Impute missing values
numerical_col_name = []
categorical_col_name = []
data['age'].fillna(data['age'].mean(), inplace=True)
data['embarked'].fillna(data['embarked'].mode()[0], inplace=True)

# Drop rows with missing values
data.dropna(inplace=True)

# Display the dataset after handling missing data
data.head()


### Questions for Exploration

1. How does the following affect model performance:
    * imputation 
    * dropping rows with missing values
2. What happens to the model performance if you use different imputation techniques (e.g., median, mode)?
3. How does the choice of dataset affect the handling of missing data?

## 3. Feature Engineering Examples

Next, we will create new features from the existing ones. We will also explore different techniques for feature engineering.

In [ ]:
# Create new features
data_new_features = data.copy()
data_new_features['family_size'] = data_new_features['sibsp'] + data_new_features['parch'] + 1
data_new_features['is_alone'] = (data_new_features['family_size'] == 1).astype(int)

# Any other features you can think of?

# Scale the numerical features
# <insert code here>

### Questions for Exploration

1. How do the new features affect model performance?
2. What other features can you create from the existing ones?
3. How does feature scaling (e.g., standardization, normalization) affect model performance? (Keep in mind that the data should be scaled **after** data splitting; this will require modifying the simple_model method in functionals.)

## 4. Visualizations/Data Exploration

We will visualize and explore the dataset using different techniques. We will also explore the impact of these visualizations on data interpretation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of age
plt.figure(figsize=(10, 6))
sns.histplot(data['age'], bins=30, kde=True)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

# Bar plot of survival by sex
plt.figure(figsize=(10, 6))
sns.barplot(x='sex', y='survived', data=data)
plt.title('Survival by Sex')
plt.xlabel('Sex')
plt.ylabel('Survival Rate')
plt.show()

# Pair plot of numerical features
sns.pairplot(data[['age', 'fare', 'family_size', 'survived']], hue='survived')
plt.show()


### Questions for Exploration

1. How do the visualizations help in understanding the dataset?
2. What other visualizations can you create to explore the dataset?
3. How does the choice of visualization technique affect the interpretation of the data?

## Extensions

Explore other data preprocessing techniques such as:
- Encoding categorical variables (e.g., one-hot encoding, label encoding) - i.e., if you modify the simple_model method to only include numerical features, omitting the categorical variables
- Feature selection techniques (i.e., assessing inter-feature correlation and removing )

Compare their impact on the dataset and the performance of downstream machine learning models. 
Can you identify any sources of bias in the dataset?